# Projeto Avaliativo – Módulo 1: Viagens a Serviço (2025)

Análise da camada **Gold** do pipeline, seguindo a Arquitetura Medallion (Raw → Silver → Gold).

**Antes de executar:** rode `0_criar_banco.sql`, `1_extrair.py` e `2_transformar.py`, nessa ordem, para que as camadas Raw e Silver estejam carregadas.

**Estrutura deste notebook**

1. **Parte 1 – Camada Silver:** 3 perguntas de negócio respondidas com consulta SQL, tabela e gráfico.
2. **Parte 2 – Camada Gold:** tabelas agregadas (e VIEWs) criadas com `JOIN` + `GROUP BY`, usadas para responder as outras 4 perguntas.

**Premissas**

- Só entram nas análises viagens com situação `Realizada`.
- `valor_total` = diárias + passagens + outros gastos − devolução (calculado em `2_transformar.py`).

In [ ]:
import textwrap
from decimal import Decimal

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display
from matplotlib.ticker import FuncFormatter

from banco import conectar

sns.set_theme(style="whitegrid")


def consultar_df(sql: str) -> pd.DataFrame:
    """Executa uma consulta SQL e devolve o resultado como DataFrame."""
    conn = conectar()
    try:
        with conn.cursor() as cursor:
            cursor.execute(sql)
            colunas = [coluna[0] for coluna in cursor.description]
            linhas = cursor.fetchall()
    finally:
        conn.close()

    df = pd.DataFrame(linhas, columns=colunas)
    if df.empty:
        raise ValueError(
            "A consulta não retornou dados. Confira se 1_extrair.py e "
            "2_transformar.py foram executados e se há viagens 'Realizada'."
        )

    # O psycopg2 devolve DECIMAL como Decimal: converte para número comum
    for coluna in df.columns:
        if df[coluna].map(lambda v: isinstance(v, Decimal)).any():
            df[coluna] = df[coluna].astype(float)
            if coluna.startswith("qtd"):
                df[coluna] = df[coluna].astype(int)
    return df


def executar_sql(comandos: list) -> None:
    """Executa uma lista de comandos SQL em uma única transação."""
    conn = conectar()
    try:
        with conn.cursor() as cursor:
            for comando in comandos:
                cursor.execute(comando)
        conn.commit()
    except Exception:
        conn.rollback()
        raise
    finally:
        conn.close()


def reais(valor: float, casas: int = 2) -> str:
    """Formata como moeda brasileira: R$ 1.234,56."""
    texto = f"{valor:,.{casas}f}"
    return "R$ " + texto.replace(",", "X").replace(".", ",").replace("X", ".")


def inteiro(valor: float) -> str:
    """Formata número inteiro com separador de milhar: 1.234."""
    return f"{valor:,.0f}".replace(",", ".")


def grafico_barras(df, x, y, titulo, rotulo_x, rotulo_y, legenda,
                   formato, rotulos=None, cor="#2a6f97"):
    """Gráfico de barras horizontais com título, eixos e legenda."""
    dados = df.copy()
    dados[y] = dados[y].astype(str).map(lambda t: textwrap.fill(t, 38))
    if rotulos is None:
        rotulos = [formato(valor) for valor in dados[x]]

    fig, ax = plt.subplots(figsize=(11, 0.9 * len(dados) + 2.5))
    sns.barplot(data=dados, x=x, y=y, color=cor, label=legenda, ax=ax)
    ax.bar_label(ax.containers[0], labels=rotulos, padding=4)
    ax.set_title(titulo, fontsize=14, fontweight="bold")
    ax.set_xlabel(rotulo_x)
    ax.set_ylabel(rotulo_y)
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: inteiro(v)))
    ax.margins(x=0.25)
    ax.legend(loc="lower right")
    plt.tight_layout()
    plt.show()

---
# Parte 1 – Perguntas respondidas a partir da camada Silver

## Pergunta 1: quais são os 5 órgãos com maior custo total?

In [ ]:
SQL_Q1 = """
SELECT
    nome_orgao_superior,
    COUNT(*)         AS qtd_viagens,
    SUM(valor_total) AS custo_total
FROM silver_viagem
WHERE situacao = 'Realizada'
GROUP BY nome_orgao_superior
ORDER BY custo_total DESC
LIMIT 5;
"""

df_q1 = consultar_df(SQL_Q1)
display(df_q1)

lider = df_q1.iloc[0]
print(
    f"Resposta: {lider['nome_orgao_superior']} tem o maior custo total, "
    f"{reais(lider['custo_total'])}, em {inteiro(lider['qtd_viagens'])} viagens."
)

grafico_barras(
    df_q1, x="custo_total", y="nome_orgao_superior",
    titulo="Top 5 órgãos superiores por custo total de viagens",
    rotulo_x="Custo total (R$)", rotulo_y="Órgão superior",
    legenda="Custo total (R$)", formato=reais,
)

**Análise:** _escreva aqui, com base no resultado acima: quanto o primeiro colocado representa em relação aos demais, se o custo está concentrado em poucos órgãos e se o número de viagens explica o custo._

## Pergunta 2: quais são os 3 destinos com maior custo médio por viagem?

O destino vem dos trechos (`silver_trecho`). Para não contar a mesma viagem duas vezes no mesmo destino, usa-se `DISTINCT` por viagem e destino. Atenção: em viagens com vários destinos, o custo total da viagem é atribuído a cada um deles.

In [ ]:
SQL_Q2 = """
SELECT
    CONCAT_WS('/', x.destino_cidade, NULLIF(x.destino_uf, '')) AS destino,
    COUNT(*)           AS qtd_viagens,
    AVG(x.valor_total) AS custo_medio
FROM (
    SELECT DISTINCT
        t.id_viagem, t.destino_cidade, t.destino_uf, v.valor_total
    FROM silver_trecho t
    JOIN silver_viagem v ON v.id_viagem = t.id_viagem
    WHERE v.situacao = 'Realizada'
      AND NULLIF(TRIM(t.destino_cidade), '') IS NOT NULL
) x
GROUP BY x.destino_cidade, x.destino_uf
ORDER BY custo_medio DESC
LIMIT 3;
"""

df_q2 = consultar_df(SQL_Q2)
display(df_q2)

for posicao, linha in df_q2.iterrows():
    print(
        f"{posicao + 1}º: {linha['destino']} – custo médio de "
        f"{reais(linha['custo_medio'])} por viagem "
        f"({inteiro(linha['qtd_viagens'])} viagens)."
    )

grafico_barras(
    df_q2, x="custo_medio", y="destino",
    titulo="Top 3 destinos por custo médio por viagem",
    rotulo_x="Custo médio por viagem (R$)", rotulo_y="Destino (cidade/UF)",
    legenda="Custo médio (R$)", formato=reais, cor="#2a9d8f",
)

**Análise:** _escreva aqui: observe a coluna `qtd_viagens`. Destinos com poucas viagens podem ter média puxada por casos isolados (por exemplo, viagens internacionais). Comente se o ranking reflete destinos realmente caros ou poucas ocorrências._

## Pergunta 3: qual a viagem de maior duração e qual foi o seu custo total?

A tabela mostra as 5 viagens mais longas; a primeira linha é a resposta. A duração é `data_fim − data_início`, calculada na Silver.

In [ ]:
SQL_Q3 = """
SELECT
    id_viagem,
    nome_orgao_superior,
    data_inicio,
    data_fim,
    duracao_dias,
    valor_total
FROM silver_viagem
WHERE situacao = 'Realizada'
  AND duracao_dias IS NOT NULL
ORDER BY duracao_dias DESC, valor_total DESC
LIMIT 5;
"""

df_q3 = consultar_df(SQL_Q3)
display(df_q3)

maior = df_q3.iloc[0]
print(
    f"Resposta: a viagem {maior['id_viagem']} ({maior['nome_orgao_superior']}) "
    f"durou {inteiro(maior['duracao_dias'])} dias e custou "
    f"{reais(maior['valor_total'])}."
)

rotulos_q3 = [
    f"{inteiro(dias)} dias | {reais(custo)}"
    for dias, custo in zip(df_q3["duracao_dias"], df_q3["valor_total"])
]
grafico_barras(
    df_q3, x="duracao_dias", y="id_viagem",
    titulo="As 5 viagens de maior duração (e seu custo total)",
    rotulo_x="Duração (dias)", rotulo_y="Identificador da viagem",
    legenda="Duração (dias)", formato=inteiro, rotulos=rotulos_q3,
    cor="#e76f51",
)

**Análise:** _escreva aqui: a viagem mais longa também é a mais cara? Compare o custo dela com o das demais da lista._

---
# Parte 2 – Camada Gold

A Gold é construída com `JOIN` + `GROUP BY`, em duas tabelas agregadas por tema, cada uma disponível **como tabela** (`gold_*`) e **como VIEW** (`vw_gold_*`):

| Objeto | Origem (JOIN) | Agrupamento |
|---|---|---|
| `gold_pagamentos` / `vw_gold_pagamentos` | `silver_pagamento` + `silver_viagem` | órgão superior, órgão pagador e tipo de pagamento |
| `gold_trechos` / `vw_gold_trechos` | `silver_trecho` + `silver_viagem` | meio de transporte e UF de destino |

A VIEW guarda a regra de negócio e sempre reflete a Silver; a tabela é uma cópia materializada dessa VIEW (`CREATE TABLE AS`), então ambas usam a mesma lógica. As agregações são separadas por tema de propósito: juntar pagamentos e trechos na mesma consulta multiplicaria os valores de cada viagem.

In [ ]:
SQL_GOLD = [
    "DROP VIEW IF EXISTS vw_gold_pagamentos;",
    "DROP TABLE IF EXISTS gold_pagamentos;",
    "DROP VIEW IF EXISTS vw_gold_trechos;",
    "DROP TABLE IF EXISTS gold_trechos;",
    """
    CREATE VIEW vw_gold_pagamentos AS
    SELECT
        v.nome_orgao_superior,
        p.nome_orgao_pagador,
        p.tipo_pagamento,
        COUNT(p.valor) AS qtd_pagamentos,
        SUM(p.valor)   AS valor_total,
        AVG(p.valor)   AS valor_medio
    FROM silver_pagamento p
    JOIN silver_viagem v ON v.id_viagem = p.id_viagem
    WHERE v.situacao = 'Realizada'
    GROUP BY v.nome_orgao_superior, p.nome_orgao_pagador, p.tipo_pagamento;
    """,
    "CREATE TABLE gold_pagamentos AS SELECT * FROM vw_gold_pagamentos;",
    """
    CREATE VIEW vw_gold_trechos AS
    SELECT
        t.meio_transporte,
        t.destino_uf,
        COUNT(*)                    AS qtd_trechos,
        COUNT(DISTINCT t.id_viagem) AS qtd_viagens,
        SUM(t.numero_diarias)       AS total_diarias
    FROM silver_trecho t
    JOIN silver_viagem v ON v.id_viagem = t.id_viagem
    WHERE v.situacao = 'Realizada'
    GROUP BY t.meio_transporte, t.destino_uf;
    """,
    "CREATE TABLE gold_trechos AS SELECT * FROM vw_gold_trechos;",
]

executar_sql(SQL_GOLD)
print("Camada Gold criada: gold_pagamentos, gold_trechos e suas VIEWs.")

display(consultar_df(
    "SELECT * FROM gold_pagamentos ORDER BY valor_total DESC LIMIT 5;"
))
display(consultar_df(
    "SELECT * FROM gold_trechos ORDER BY qtd_trechos DESC LIMIT 5;"
))

## Pergunta 4: qual o tipo de pagamento com maior valor médio?

O valor médio por tipo é calculado como `SUM(valor_total) / SUM(qtd_pagamentos)`, o que evita a distorção de tirar a média das médias de cada grupo.

In [ ]:
SQL_Q4 = """
SELECT
    tipo_pagamento,
    SUM(qtd_pagamentos)                             AS qtd_pagamentos,
    SUM(valor_total) / NULLIF(SUM(qtd_pagamentos), 0) AS valor_medio
FROM gold_pagamentos
GROUP BY tipo_pagamento
ORDER BY valor_medio DESC;
"""

df_q4 = consultar_df(SQL_Q4)
display(df_q4)

topo = df_q4.iloc[0]
print(
    f"Resposta: o tipo de pagamento com maior valor médio é "
    f"{topo['tipo_pagamento']}, com {reais(topo['valor_medio'])} por "
    f"pagamento ({inteiro(topo['qtd_pagamentos'])} pagamentos)."
)

grafico_barras(
    df_q4, x="valor_medio", y="tipo_pagamento",
    titulo="Valor médio por tipo de pagamento",
    rotulo_x="Valor médio por pagamento (R$)", rotulo_y="Tipo de pagamento",
    legenda="Valor médio (R$)", formato=reais, cor="#6a4c93",
)

**Análise:** _escreva aqui: compare o valor médio com a quantidade de pagamentos de cada tipo; um tipo pode ter média alta e ser raro._

## Pergunta 5: qual o meio de transporte mais usado nos trechos?

In [ ]:
SQL_Q5 = """
SELECT
    meio_transporte,
    SUM(qtd_trechos) AS qtd_trechos
FROM gold_trechos
WHERE NULLIF(TRIM(meio_transporte), '') IS NOT NULL
GROUP BY meio_transporte
ORDER BY qtd_trechos DESC;
"""

df_q5 = consultar_df(SQL_Q5)
display(df_q5)

total_trechos = df_q5["qtd_trechos"].sum()
topo = df_q5.iloc[0]
print(
    f"Resposta: {topo['meio_transporte']} é o meio mais usado, com "
    f"{inteiro(topo['qtd_trechos'])} trechos "
    f"({topo['qtd_trechos'] / total_trechos:.1%} do total)."
)

grafico_barras(
    df_q5, x="qtd_trechos", y="meio_transporte",
    titulo="Quantidade de trechos por meio de transporte",
    rotulo_x="Quantidade de trechos", rotulo_y="Meio de transporte",
    legenda="Trechos", formato=inteiro, cor="#f4a261",
)

**Análise:** _escreva aqui: qual a participação do meio mais usado e o que isso diz sobre o perfil das viagens._

## Pergunta 6: qual UF de destino aparece em mais trechos?

O gráfico mostra as 10 UFs com mais trechos.

In [ ]:
SQL_Q6 = """
SELECT
    destino_uf,
    SUM(qtd_trechos) AS qtd_trechos
FROM gold_trechos
WHERE NULLIF(TRIM(destino_uf), '') IS NOT NULL
GROUP BY destino_uf
ORDER BY qtd_trechos DESC
LIMIT 10;
"""

df_q6 = consultar_df(SQL_Q6)
display(df_q6)

topo = df_q6.iloc[0]
print(
    f"Resposta: {topo['destino_uf']} é a UF de destino mais frequente, "
    f"com {inteiro(topo['qtd_trechos'])} trechos."
)

grafico_barras(
    df_q6, x="qtd_trechos", y="destino_uf",
    titulo="Top 10 UFs de destino por quantidade de trechos",
    rotulo_x="Quantidade de trechos", rotulo_y="UF de destino",
    legenda="Trechos", formato=inteiro, cor="#264653",
)

**Análise:** _escreva aqui: o destino mais frequente é o esperado? Há relação com a sede dos órgãos federais?_

## Pergunta 7: qual órgão pagou mais no total?

Aqui a consulta usa a **VIEW** `vw_gold_pagamentos`, agrupando pelo órgão pagador. Note que é diferente da Pergunta 1, que olha o órgão superior ao qual a viagem pertence.

In [ ]:
SQL_Q7 = """
SELECT
    nome_orgao_pagador,
    SUM(qtd_pagamentos) AS qtd_pagamentos,
    SUM(valor_total)    AS total_pago
FROM vw_gold_pagamentos
GROUP BY nome_orgao_pagador
ORDER BY total_pago DESC
LIMIT 5;
"""

df_q7 = consultar_df(SQL_Q7)
display(df_q7)

topo = df_q7.iloc[0]
print(
    f"Resposta: {topo['nome_orgao_pagador']} foi o órgão que mais pagou, "
    f"{reais(topo['total_pago'])} em {inteiro(topo['qtd_pagamentos'])} pagamentos."
)

**Análise:** _escreva aqui: o órgão que mais pagou coincide com o de maior custo total na Pergunta 1? Se não, explique a diferença entre órgão superior e órgão pagador._

---
# Conclusões e limitações

**Principais conclusões:** _resuma aqui, em 3 ou 4 tópicos, os achados das perguntas acima._

**Limitações e decisões de análise**

- Foram consideradas apenas viagens com situação `Realizada`; incluir outras situações altera os totais.
- `duracao_dias` é a diferença entre as datas (uma viagem de um único dia tem duração 0).
- Na Pergunta 2, o custo de viagens com vários destinos é atribuído a cada destino.
- Os dados cobrem o recorte de 6 meses de 2025 fornecido para o projeto.

**Possíveis melhorias:** carga incremental por período, validação de qualidade dos dados na camada Silver e dashboards interativos sobre as VIEWs da Gold.